# Workflow 07_B: Tool-Augmented RAG Baseline (1 tool call + 1 LLM call)

**Architecture**
1. Receive user query.
2. Rule-based router selects one tool: `product_search`, `policy_search`, or `order_lookup` (if auth token provided).
3. Call the selected tool once with simple parameters.
4. Feed tool output (context) to a single `LLMService.call_gemini` call.
5. Return the LLM answer.

**Limitation vs Agentic RAG**
- No multi-turn planning or context reuse.
- No tool loops (e.g., compare needs two product searches).
- Router is rule-based, not learned.


In [ ]:
# Setup path and imports
import sys, os, json, time
from pathlib import Path

notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

from configs.setting import settings
from configs.GetConfig import config
from src.LLMService import LLMService
from src.h_evaluation.benchmark_evaluator import BenchmarkEvaluator
from src.d_tools.product.product_search import product_search
from src.d_tools.policy.policy_search import policy_search
from src.d_tools.account.order_lookup import order_lookup
import re

llm_service = LLMService(settings, config)
model = config.llm.google.available[0]

# Fill in if you want to test order queries
USER_TOKEN = None
USER_ID = None
print("Model:", model)


In [ ]:
def extract_text(response):
    """Extract text from LLMService response (list of chunks or single response)."""
    if not response:
        return ""
    if isinstance(response, list):
        parts = []
        for chunk in response:
            if getattr(chunk, "text", None):
                parts.append(chunk.text)
            elif getattr(chunk, "candidates", None):
                for cand in chunk.candidates:
                    content = getattr(cand, "content", None)
                    if content:
                        for part in getattr(content, "parts", []):
                            parts.append(getattr(part, "text", ""))
        return "".join(parts)
    return getattr(response, "text", "")

def select_tool(query: str, has_auth: bool) -> str:
    """Rule-based tool router."""
    q = query.lower()
    if has_auth and any(k in q for k in ["đơn hàng", "order", "tra đơn", "mua hàng của tôi"]):
        return "order_lookup"
    if any(k in q for k in ["chính sách", "đổi trả", "bảo hành", "trả góp", "giao hàng"]):
        return "policy_search"
    return "product_search"


def tool_augmented_answer(query, has_auth=False, user_token=None, user_id=None):
    """Single-tool then single LLM call."""
    t0 = time.time()
    tool_name = select_tool(query, has_auth)
    tool_outputs = []
    actual_tool_calls = []

    try:
        if tool_name == "product_search":
            q = query.lower()
            include_details = any(k in q for k in ["ram", "chip", "cpu", "pin", "màn hình", "camera", "bộ nhớ", "thông số", "cấu hình"])
            need_price = any(k in q for k in ["giá", "tồn kho", "stock", "bao nhiêu tiền", "khuyến mãi"])
            ctx = product_search(queries=[{
                "keyword": query,
                "limit": 3,
                "include_details": include_details,
                "need_price_info": need_price,
            }])
        elif tool_name == "policy_search":
            kw = re.sub(r"[^\w\s]", "", query).strip()[:60]
            ctx = policy_search(key_word=kw, limit=3)
        elif tool_name == "order_lookup" and has_auth:
            ctx = order_lookup(current_user_id=user_id, user_token=user_token, order_id=None)
        else:
            ctx = "Không đủ thông tin để tra cứu."
    except Exception as e:
        ctx = f"Lỗi khi gọi tool {tool_name}: {e}"

    tool_outputs.append(ctx)
    actual_tool_calls.append({"tool": tool_name, "args": {"query": query}})

    messages = [
        {"role": "system", "content": "Bạn là trợ lý bán hàng. Dùng thông tin TOOL OUTPUT bên dưới để trả lời câu hỏi. Nếu thông tin không đủ, hãy nói rõ."},
        {"role": "user", "content": f"TOOL OUTPUT:\n{ctx[:4000]}\n\nCÂU HỎI: {query}\n\nTrả lời:"}
    ]
    response = llm_service.call_gemini(model=model, messages=messages, tools=None, stream=False)
    answer = extract_text(response)
    return {
        "final_answer": answer,
        "tool_outputs": tool_outputs,
        "actual_tool_calls": actual_tool_calls,
        "latency": time.time() - t0,
        "total_tokens": 0,
    }


In [ ]:
# Test on 3 sample queries
bench_path = Path("/home/ubuntu/benchmark_results/ecommerce_benchmark_20each_1785642048.jsonl")
samples = [json.loads(l) for l in bench_path.read_text(encoding='utf-8').splitlines()[:3]]

for rec in samples:
    q = rec.get("question", "") or rec["turns"][0]["question"]
    print(f"\nQ: {q}")
    res = tool_augmented_answer(q, has_auth=bool(USER_TOKEN), user_token=USER_TOKEN, user_id=USER_ID)
    print(f"Tool: {res['actual_tool_calls'][0]['tool']}")
    print(f"A: {res['final_answer'][:300]}...")


In [ ]:
def flatten_benchmark(path, limit=None):
    rows = []
    with open(path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            rec = json.loads(line)
            if rec.get("turns"):
                for j, turn in enumerate(rec["turns"]):
                    rows.append({
                        "id": f"{rec['id']}_turn{j+1}",
                        "category": rec.get("category", ""),
                        "question": turn["question"],
                        "expected_tool_calls": turn.get("expected_tool_calls", []),
                        "ground_truth": turn.get("ground_truth", {}),
                    })
            else:
                rows.append({
                    "id": rec["id"],
                    "category": rec.get("category", ""),
                    "question": rec["question"],
                    "expected_tool_calls": rec.get("expected_tool_calls", []),
                    "ground_truth": rec.get("ground_truth", {}),
                })
    return rows

# Run on first N records and save raw results
BENCH_LIMIT = 10

raw_results = []
for rec in flatten_benchmark("/home/ubuntu/benchmark_results/ecommerce_benchmark_20each_1785642048.jsonl", limit=BENCH_LIMIT):
    try:
        res = tool_augmented_answer(rec["question"], has_auth=bool(USER_TOKEN), user_token=USER_TOKEN, user_id=USER_ID)
        raw_results.append({
            "id": rec["id"],
            "category": rec["category"],
            "question": rec["question"],
            "workflow": "07_B_tool_rag",
            "final_answer": res["final_answer"],
            "tool_outputs": res["tool_outputs"],
            "actual_tool_calls": res["actual_tool_calls"],
            "ground_truth": rec["ground_truth"],
            "expected_tool_calls": rec["expected_tool_calls"],
            "latency": res["latency"],
            "total_tokens": res["total_tokens"],
        })
    except Exception as e:
        print(f"Error on {rec['id']}: {e}")

out_path = Path("benchmark_results/raw_07_B_tool_rag.jsonl")
out_path.parent.mkdir(exist_ok=True, parents=True)
with open(out_path, "w", encoding="utf-8") as f:
    for r in raw_results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(raw_results)} rows to {out_path}")


In [ ]:
# Evaluate the raw results (custom judge only; set use_ragas=True if API quota allows)
evaluator = BenchmarkEvaluator(judge_provider='groq', use_ragas=False)
report = evaluator.evaluate(raw_results, output_dir='benchmark_results')
evaluator.print_table(report)
